In [ ]:
import pandas as pd
import numpy as np

### The official backend vector layer URL (discovered via developer tools)
URL = "https://fire.boku.ac.at/firedb/webgis/GetWaldbrandData"

In [2]:
# Target file path matching today's compiled layout index
file_path = "../data/boku_waldbrand_02-11-2026.csv"

try:
    # Load the CSV data securely into a structured Dataframe matrix
    # Using utf-8-sig to cleanly render native Austrian text characters
    df = pd.read_csv(file_path, encoding="utf-8-sig")

    print("-" * 40)
    print("DATAFRAME:")
    print("-" * 40)
    
    # 1. Print out the precise shape matrix metrics (Rows, Columns)
    print(f"Shape of Dataframe (Rows, Columns): {df.shape}")
    print(f"  --> Total recorded fire incidents:  {df.shape[0]}")
    print(f"  --> Total tracking variables:       {df.shape[1]}")
    print("-" * 40)

    # 2. Print out the formal database column headers
    print("COLUMNS:")
    print(df.columns.tolist())
    print("-" * 40)

    # 3. Print out a random sample slice of 10 rows
    print("SAMPLE ROWS:")
    # Selecting a subset of key columns for cleaner on-screen terminal viewing
    preview_cols = ["waldbrandid", "jahr", "monat", "tag", "bundesland", "gemeinde_neu", "brandflaeche"]
    
    # Fallback to standard head if columns differ in your custom file structure
    cols_to_show = [c for c in preview_cols if c in df.columns] or df.columns
    
    print(df[cols_to_show].sample(n=10, random_state=17))
    print("=" * 40)
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found in your working directory.")
    print("Please verify your local file naming structure matching today's extraction.")
except Exception as e:
    print(f"System error reading data: {e}")


----------------------------------------
DATAFRAME:
----------------------------------------
Shape of Dataframe (Rows, Columns): (6679, 19)
  --> Total recorded fire incidents:  6679
  --> Total tracking variables:       19
----------------------------------------
COLUMNS:
['objid', 'waldbrandid', 'jahr', 'monat', 'tag', 'bundesland', 'gemeinde', 'brandart', 'brandflaeche', 'ursache1', 'feuerwehren', 'einsatzkraefte', 'quelle1', 'verifiziert', 'gemeinde_neu', 'brandflaeche_korr', 'schutzwald', 'longitude', 'latitude']
----------------------------------------
SAMPLE ROWS:
      waldbrandid  jahr  monat  tag bundesland                 gemeinde_neu  \
4162   2018081401  2018      8   14          K          Ebenthal in Kärnten   
96     2008110201  2008     11    2        NOE                     Wölbling   
4030   2017032703  2017      3   27          K          Ebenthal in Kärnten   
2373   2015070701  2015      7    7        NOE                   Scharndorf   
4009   2016010201  2016    

### Separate numeric and categorical elements

In [3]:

    # Clean the dataset: replace standard BOKU missing data flags (-9999) with real NaN values
    # This prevents -9999 from distorting your true minimum range values
    df_cleaned = df.replace([-9999, -9999.0], np.nan)

    print("-" * 40)
    print("DATA PROFILE:")
    print("-" * 40)

    # Separate numeric and categorical elements
    numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns.tolist()
    text_cols = df_cleaned.select_dtypes(include=["object", "category"]).columns.tolist()

    print(f"NUMERIC COLUMNS (Min - Max)")
    print("-" * 40)
    for col in numeric_cols:
        col_min = df_cleaned[col].min()
        col_max = df_cleaned[col].max()

        # Format outputs cleanly based on integer vs float structures
        if pd.isna(col_min) or pd.isna(col_max):
            print(f"-{col:<20} [empty]")
        elif df_cleaned[col].dropna().round().isin([df_cleaned[col].dropna()]).all():
            print(f"-{col:<20} Min: {int(col_min):, samples} | Max: {int(col_max):, samples}")
        else:
            print(f"-{col:<20} Min: {col_min:,.4f} | Max: {col_max:,.4f}")

    print("\n" + "-" * 40)
    print(f"TEXT COLUMNS")
    print("-" * 40)
    for col in text_cols:
        # Drop empty cells and fetch unique strings sorted alphabetically
        unique_values = sorted(df_cleaned[col].dropna().unique().astype(str))
        total_unique = len(unique_values)

        if total_unique == 0:
            print(f"{col:<20} [Empty]")
        elif total_unique <= 15:
            # If the set is small (like Federal Provinces or Fire Types), show the full list
            values_str = ", ".join([f"'{v}'" for v in unique_values])
            print(f"{col:<20} ({total_unique} categories) \n    {{ {values_str} }}")
        else:
            # If the set is huge (like Municipalities or Unique IDs), show a preview snippet
            preview_str = ", ".join([f"'{v}'" for v in unique_values[:5]])
            print(
                f"{col:<20} ({total_unique} categories) \n    {{ {preview_str}, ... [+{total_unique-5} more values] }}"
            )

    print("-" * 40)


----------------------------------------
DATA PROFILE:
----------------------------------------
NUMERIC COLUMNS (Min - Max)
----------------------------------------
-objid                Min: 7.0000 | Max: 81,293.0000
-waldbrandid          Min: 2,002,010,101.0000 | Max: 2,026,101,701.0000
-jahr                 Min: 2,002.0000 | Max: 2,026.0000
-monat                Min: 1.0000 | Max: 12.0000
-tag                  Min: 1.0000 | Max: 31.0000
-brandart             Min: 0.0000 | Max: 7.0000
-brandflaeche         Min: 1.0000 | Max: 8,329,060.0000
-ursache1             Min: 0.0000 | Max: 2.0000
-feuerwehren          Min: 0.0000 | Max: 82.0000
-einsatzkraefte       Min: 0.0000 | Max: 1,000.0000
-brandflaeche_korr    Min: -99,990,000.0000 | Max: 1,851,260.0000
-schutzwald           Min: 0.0000 | Max: 2.0000
-longitude            Min: 9.5680 | Max: 17.1310
-latitude             Min: 46.4390 | Max: 48.9940

----------------------------------------
TEXT COLUMNS
-----------------------------------